In [0]:
# Gold Layer: Daily Sales Summary

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, sum as spark_sum, count, approx_count_distinct,
    current_timestamp, lit, round as spark_round, when, coalesce
)

spark = SparkSession.builder.getOrCreate()
GOLD_TABLE = "ecomstore.ecomlake.gold_daily_sales_summary"

orders = spark.read.table("ecomstore.ecomlake.silver_orders")
returns = spark.read.table("ecomstore.ecomlake.silver_returns")
payments = spark.read.table("ecomstore.ecomlake.silver_payments")

# PRE-AGGREGATE DAILY RETURNS
daily_returns = (
    returns.alias("r")
    .join(orders.alias("o"), on="order_id", how="inner")
    .groupBy("o.order_date")
    .agg(spark_sum("r.refund_amount").alias("daily_refunds"))
)

# PRE-AGGREGATE DAILY PAYMENTS
daily_payments = (
    payments.alias("p")
    .join(orders.alias("o"), on="order_id", how="inner")
    .groupBy("o.order_date")
    .agg(
        spark_sum(when(col("p.payment_status") == "success", col("p.amount")).otherwise(0)).alias("actual_revenue_collected"),
        count(when(col("p.payment_status") == "failed", True)).alias("failed_payment_attempts"),
        count(when(col("p.payment_method") == "upi", True)).alias("upi_transactions_count")
    )
)

# BUILD DAILY SALES SUMMARY
daily_summary = (
    orders
    .filter(col("status").isin(["delivered", "shipped", "out_for_delivery", "processing", "confirmed", "cancelled"]))
    .filter(col("order_date").isNotNull())
    .groupBy("order_date")
    .agg(
        count("order_id").alias("total_orders"),
        approx_count_distinct("customer_id").alias("unique_customers"),
        
        # Gross metrics[cite: 1]
        spark_sum("final_amount").alias("gross_merchandise_value"),
        spark_sum(when(col("status") == "cancelled", col("final_amount")).otherwise(0)).alias("cancelled_revenue"),
        spark_sum("discount_amount").alias("total_discount_given"),
        
        # Operational metrics[cite: 1]
        count(when(col("status") == "delivered", True)).alias("delivered_orders"),
        count(when(col("status") == "cancelled", True)).alias("cancelled_orders")
    )
    # Join returns & payments
    .join(daily_returns, on="order_date", how="left")
    .join(daily_payments, on="order_date", how="left")
    .withColumn("daily_refunds", coalesce(col("daily_refunds"), lit(0.0)))
    .withColumn("actual_revenue_collected", coalesce(col("actual_revenue_collected"), lit(0.0)))
    
    # Mathematical Accuracy: True Net Revenue
    .withColumn("net_merchandise_value", 
        spark_round(col("gross_merchandise_value") - col("cancelled_revenue") - col("daily_refunds"), 2)
    )
    .withColumn("cancellation_rate",
        when(col("total_orders") > 0, spark_round(col("cancelled_orders") / col("total_orders") * 100, 2))
        .otherwise(0.0)
    )
    .withColumn("payment_failure_rate",
        when(col("total_orders") > 0, spark_round(col("failed_payment_attempts") / col("total_orders") * 100, 2))
        .otherwise(0.0)
    )
    .withColumn("_gold_processed_at", current_timestamp())
    .orderBy("order_date")
)

# 4. WRITE TO GOLD
(
    daily_summary
    .coalesce(1)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.autoOptimize.optimizeWrite", "true")
    .option("delta.autoOptimize.autoCompact", "true")
    .partitionBy("order_date")
    .saveAsTable(GOLD_TABLE)
)